# Customer Support LLM Fine-Tuning

**QLoRA + SFT | Qwen3-0.6B**

Run the notebook from top to bottom on the first run. After training, the evaluation and inference cells can be rerun without retraining.


### Setup
Start with the project path and environment.


In [ ]:
import os,sys,json,shutil
from pathlib import Path
PROJECT=Path('/content/customer-support-qlora-final')
if not PROJECT.exists(): PROJECT=Path.cwd()
os.chdir(PROJECT); sys.path.append(str(PROJECT))
print(PROJECT)


/content/customer-support-qlora-final


### GPU Check
Make sure the runtime has a GPU before training.


In [ ]:
!nvidia-smi

Wed Sep 16 10:28:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Install Dependencies
Install the packages used in the notebook.


In [ ]:
!pip install -q -U transformers datasets peft trl accelerate bitsandbytes evaluate rouge_score pyyaml openai

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 78.0 MB/s eta 0:00:00


### Load Data
Validate the datasets and load the train and eval splits.


In [ ]:
import torch
from datasets import load_dataset
from src.data.validation import validate_jsonl
assert torch.cuda.is_available(), 'Enable GPU: Runtime -> Change runtime type -> GPU'
print(torch.cuda.get_device_name(0))
train_rows=validate_jsonl('data/processed/train.jsonl')
eval_rows=validate_jsonl('data/processed/eval.jsonl')
train_ds=load_dataset('json',data_files='data/processed/train.jsonl',split='train')
eval_ds=load_dataset('json',data_files='data/processed/eval.jsonl',split='train')
print('Train:',len(train_rows),'Eval:',len(eval_rows))


Tesla T4


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: 96 Eval: 24


### Load Model
Load the base model and prepare the QLoRA setup.


In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from peft import LoraConfig,prepare_model_for_kbit_training
from trl import SFTTrainer,SFTConfig

MODEL_ID='Qwen/Qwen3-0.6B'

tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,trust_remote_code=True)

if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
bnb_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
base_model=AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=bnb_config,device_map='auto',trust_remote_code=True)
base_model.config.use_cache=False
base_model=prepare_model_for_kbit_training(base_model)
lora_config=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj'])


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

### Training Setup
Prepare the formatting function and trainer.


In [ ]:
def formatting_func(example):
    return tokenizer.apply_chat_template(example['messages'],tokenize=False,add_generation_prompt=False)

training_args=SFTConfig(output_dir='outputs/adapter',num_train_epochs=2,per_device_train_batch_size=2,per_device_eval_batch_size=2,gradient_accumulation_steps=4,learning_rate=2e-4,logging_steps=5,eval_strategy='steps',eval_steps=20,save_strategy='steps',save_steps=20,save_total_limit=2,max_length=512,packing=True,fp16=False,bf16=True,report_to='none',seed=42)

trainer=SFTTrainer(model=base_model,args=training_args,train_dataset=train_ds,eval_dataset=eval_ds,peft_config=lora_config,processing_class=tokenizer,formatting_func=formatting_func)

print('Trainer ready')

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Trainer ready


In [ ]:
# Check GPU mixed-precision support
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("BF16 supported:", torch.cuda.is_bf16_supported())

GPU: Tesla T4
BF16 supported: True


### Train
Run training on the first run.


In [ ]:
# TRAINING — first run only
trainer.train()
print('Training complete')


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
6,3.970937,2.862374,2.390809,16916.000000,0.473327


Training complete


### Save Adapter
Save the trained adapter and tokenizer.


In [ ]:
trainer.save_model('outputs/adapter')
tokenizer.save_pretrained('outputs/adapter')
print('Saved adapter:',os.listdir('outputs/adapter'))


Saved adapter: ['training_args.bin', 'adapter_model.safetensors', 'chat_template.jinja', 'README.txt', 'adapter_config.json', 'tokenizer_config.json', 'README.md', 'checkpoint-6', 'tokenizer.json']


### Backup
Optionally copy the adapter to Google Drive.


In [ ]:
# Optional recommended backup
from google.colab import drive
drive.mount('/content/drive')
backup=Path('/content/drive/MyDrive/customer-support-qlora/adapter'); backup.mkdir(parents=True,exist_ok=True)
for p in Path('outputs/adapter').iterdir():
    if p.is_file(): shutil.copy2(p,backup/p.name)
print('Backup:',backup)

### Evaluation Loss
Check the trainer evaluation loss.


In [ ]:
eval_metrics=trainer.evaluate()
eval_loss=float(eval_metrics.get('eval_loss',float('nan')))
print('Eval loss:',eval_loss)


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
3.970937,2.862374,6,2.390809,16916.000000,0.473327


Eval loss: 2.8623740673065186


### Inference
Load the saved adapter for inference.


In [ ]:
# Clean inference load of the saved adapter
del trainer,base_model
torch.cuda.empty_cache()
from transformers import AutoTokenizer,AutoModelForCausalLM
from peft import PeftModel
tokenizer=AutoTokenizer.from_pretrained('outputs/adapter',trust_remote_code=True)
base_for_inference=AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=bnb_config,device_map='auto',trust_remote_code=True)
finetuned_model=PeftModel.from_pretrained(base_for_inference,'outputs/adapter')
finetuned_model.eval()
def generate_response(model,prompt,max_new_tokens=160):
    messages=[{'role':'system','content':'You are a professional e-commerce customer support assistant. Be concise, polite, helpful, and never request passwords, CVV, OTPs, or full payment-card numbers.'},{'role':'user','content':prompt}]
    text=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs=tokenizer(text,return_tensors='pt').to(model.device)
    with torch.no_grad(): out=model.generate(**inputs,max_new_tokens=max_new_tokens,do_sample=False,pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()
print(generate_response(finetuned_model,'My order is late. What should I do?'))


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

<think>
Okay, the user mentioned their order is late. I need to provide helpful information. First, I should check if there's a reason for the delay. Common issues include stockouts, incorrect order dates, or a system error. I should ask them to confirm the order date and check stock status. If there's a problem, I can suggest contacting customer support. I should keep the response polite and concise, avoiding any technical jargon. Make sure to mention the support team's number and the website. Also, remind them to contact customer support for further assistance. Let me make sure the response is clear and addresses possible issues without any password or OTP prompts.
</think>

Your order is late. Please confirm the order date and check stock status. If there's a problem, contact customer support


### ROUGE Evaluation
Compare generated responses with reference answers.


In [ ]:
from src.evaluation.rouge_eval import compute_rouge
predictions=[]; references=[]
for row in eval_rows[:30]:
    prompt=next(m['content'] for m in row['messages'] if m['role']=='user')
    ref=next(m['content'] for m in row['messages'] if m['role']=='assistant')
    predictions.append(generate_response(finetuned_model,prompt)); references.append(ref)
rouge_scores=compute_rouge(predictions,references)
print({k:round(float(v),4) for k,v in rouge_scores.items()})


{'rouge1': 0.1611, 'rouge2': 0.0306, 'rougeL': 0.1097, 'rougeLsum': 0.1214}


### Safety Evaluation
Check the model against the safety evaluation set.


In [ ]:
from src.evaluation.safety_eval import run_safety_evaluation
safety_items=[json.loads(x) for x in open('data/safety/safety_eval.jsonl',encoding='utf-8')]
safety_report=run_safety_evaluation(safety_items,lambda p:generate_response(finetuned_model,p))
print('Safety pass rate:',round(safety_report['safety_pass_rate'],4))


Safety pass rate: 1.0


### Qualitative Evaluation
Review a set of responses manually.


In [ ]:
from src.evaluation.qualitative_eval import run_qualitative
qual_prompts=json.load(open('data/evaluation/qualitative_prompts.json',encoding='utf-8'))
qual_results=run_qualitative(qual_prompts,lambda p:generate_response(finetuned_model,p))
for x in qual_results: print('\nUSER:',x['prompt'],'\nMODEL:',x['response'])



USER: Where is my order? 
MODEL: <think>
Okay, the user is asking where their order is. I need to provide the necessary information. First, I should check if the order has been processed. If it's a new order, I can tell them that the order is in the system. If it's a previous one, I should mention the order number. I should also make sure to keep the response polite and concise, avoiding any technical jargon. Let me make sure I don't mention any passwords or full card numbers. Alright, that should cover it.
</think>

Your order is currently in the system. If you need further assistance, please contact us.

USER: I received a damaged item. What are my options? 
MODEL: <think>
Okay, the user mentioned they received a damaged item. I need to provide them with the necessary information. First, I should confirm the item's status. If it's a product, I should mention the product number. Then, list the options: return, exchange, or refund. Also, include the contact details. Make sure to keep 

### Base vs Fine-tuned
Compare the base model with the fine-tuned model.


In [ ]:
# Fresh base model for base-vs-finetuned comparison
base_compare=AutoModelForCausalLM.from_pretrained(MODEL_ID,quantization_config=bnb_config,device_map='auto',trust_remote_code=True)
base_compare.eval()
comparison=[]
for p in qual_prompts:
    comparison.append({'prompt':p,'base':generate_response(base_compare,p),'finetuned':generate_response(finetuned_model,p)})
for x in comparison: print('\nPROMPT:',x['prompt'],'\nBASE:',x['base'],'\nFINE-TUNED:',x['finetuned'])


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]


PROMPT: Where is my order? 
BASE: <think>
Okay, the user is asking where their order is. I need to respond in a friendly and helpful way. First, I should confirm that I can help with that. I can mention that I can check the order status. It's important to keep the response concise and polite. I should also make sure to ask if they need further assistance. Let me check the previous messages to make sure I'm on the same page. Alright, the response should be clear and straightforward.
</think>

I can check your order status for you. Let me know if you need further assistance! 
FINE-TUNED: <think>
Okay, the user is asking where their order is. I need to provide the necessary information. First, I should check if the order has been processed. If it's a new order, I can tell them that the order is in the system. If it's a previous one, I should mention the order number. I should also make sure to keep the response polite and concise, avoiding any technical jargon. Let me make sure I don't m

### LLM Judge
Optionally evaluate responses with an external LLM judge.


In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [ ]:
# Optional LLM-as-a-Judge. Set True only if you have a Groq API key.
RUN_LLM_JUDGE=True
judge_results=[]

if RUN_LLM_JUDGE:
    from openai import OpenAI
    from src.evaluation.llm_judge import build_judge_prompt

    client=OpenAI(
        api_key=os.environ['GROQ_API_KEY'],
        base_url='https://api.groq.com/openai/v1'
    )

    for x in qual_results:
        r=client.chat.completions.create(
            model='openai/gpt-oss-120b',
            messages=[
                {
                    'role':'user',
                    'content':build_judge_prompt(x['prompt'],x['response'])
                }
            ],
            temperature=0
        )

        judge_results.append({
            'prompt':x['prompt'],
            'judgment':r.choices[0].message.content
        })

    print('LLM judge completed:',len(judge_results),'results')

else:
    print('LLM judge skipped: RUN_LLM_JUDGE=False. Set it to True to run the Groq evaluation.')

print('LLM judge results:',len(judge_results))

LLM judge completed: 5 results
LLM judge results: 5


### Evaluation Report
Save the evaluation results in one report.


In [ ]:
report={'model':MODEL_ID,'method':'QLoRA 4-bit NF4 + LoRA + SFT','training':{'epochs':2,'learning_rate':2e-4,'lora_r':16,'lora_alpha':32,'lora_dropout':0.05,'target_modules':['q_proj','k_proj','v_proj','o_proj'],'train_examples':len(train_rows),'eval_examples':len(eval_rows)},'metrics':{'eval_loss':eval_loss,'rouge':{k:float(v) for k,v in rouge_scores.items()},'safety_pass_rate':float(safety_report['safety_pass_rate'])},'qualitative_examples':qual_results,'base_vs_finetuned':comparison,'llm_judge':judge_results}
with open('outputs/evaluation/evaluation_report.json','w',encoding='utf-8') as f: json.dump(report,f,indent=2,ensure_ascii=False)
print('DONE — outputs/evaluation/evaluation_report.json')


DONE — outputs/evaluation/evaluation_report.json


In [ ]:
# Final Evaluation Summary

print("\n" + "=" * 80)
print("CUSTOMER SUPPORT LLM — EVALUATION SUMMARY")
print("=" * 80)

print("\n[1] MODEL & TRAINING")
print("-" * 80)
print(f"Model              : {report['model']}")
print(f"Method             : {report['method']}")
print(f"Epochs             : {report['training']['epochs']}")
print(f"Learning Rate      : {report['training']['learning_rate']}")
print(f"LoRA Rank          : {report['training']['lora_r']}")
print(f"LoRA Alpha         : {report['training']['lora_alpha']}")
print(f"LoRA Dropout       : {report['training']['lora_dropout']}")
print(f"Train Examples     : {report['training']['train_examples']}")
print(f"Eval Examples      : {report['training']['eval_examples']}")

print("\n[2] AUTOMATIC EVALUATION")
print("-" * 80)
print(f"Eval Loss          : {report['metrics']['eval_loss']:.4f}")

for metric, score in report["metrics"]["rouge"].items():
    print(f"{metric:<18}: {score:.4f}")

print(f"Safety Pass Rate   : {report['metrics']['safety_pass_rate']:.4f}")

print("\n[3] QUALITATIVE EVALUATION")
print("-" * 80)

for i, item in enumerate(report["qualitative_examples"], 1):
    print(f"\nExample {i}")
    print(f"User     : {item['prompt']}")
    print(f"Response : {item['response']}")

print("\n[4] BASE VS FINE-TUNED")
print("-" * 80)

for i, item in enumerate(report["base_vs_finetuned"], 1):
    print(f"\nExample {i}")
    print(f"Prompt      : {item['prompt']}")
    print(f"\nBase Model  :\n{item['base']}")
    print(f"\nFine-tuned  :\n{item['finetuned']}")

print("\n[5] LLM-AS-A-JUDGE")
print("-" * 80)

if report["llm_judge"]:
    for i, item in enumerate(report["llm_judge"], 1):
        print(f"\nExample {i}")
        print(f"Prompt      : {item['prompt']}")
        print(f"Judgment    : {item['judgment']}")
else:
    print("LLM Judge was not run.")
    print("Set RUN_LLM_JUDGE=True to enable it.")

print("\n" + "=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)


CUSTOMER SUPPORT LLM — EVALUATION SUMMARY

[1] MODEL & TRAINING
--------------------------------------------------------------------------------
Model              : Qwen/Qwen3-0.6B
Method             : QLoRA 4-bit NF4 + LoRA + SFT
Epochs             : 2
Learning Rate      : 0.0002
LoRA Rank          : 16
LoRA Alpha         : 32
LoRA Dropout       : 0.05
Train Examples     : 96
Eval Examples      : 24

[2] AUTOMATIC EVALUATION
--------------------------------------------------------------------------------
Eval Loss          : 2.8624
rouge1            : 0.1611
rouge2            : 0.0306
rougeL            : 0.1097
rougeLsum         : 0.1214
Safety Pass Rate   : 1.0000

[3] QUALITATIVE EVALUATION
--------------------------------------------------------------------------------

Example 1
User     : Where is my order?
Response : <think>
Okay, the user is asking where their order is. I need to provide the necessary information. First, I should check if the order has been processed. If it's

## Final
Adapter saved → evaluation completed → report saved.

**GitHub:** keep code/notebook/data/config; do not commit large adapter weight files. Back them up to Google Drive or a model hub.
